# Kiểm tra nhanh: dataset video EK100 có phải bản gốc không? (~5 phút, không cần GPU)

*Add Input*: dataset video EK100 **và** output lần chạy `kaggle_verify_ek100` (thư mục có `logits/official`).

- Bản gốc của EK100: **1920x1080**, bitrate cao (thường > 10 Mbit/s), 138 video val nặng tổng cộng vài chục GB trở lên.
- Nếu dataset chỉ khoảng ≤ 20GB, hoặc độ phân giải thấp hơn, hoặc bitrate thấp, thì video đã bị encode lại, và điểm sẽ bị kéo xuống.

In [ ]:
BRANCH = "claude/vjepa2-verify-improve-dsfgtg"
VJEPA2_COMMIT = "204698b45b3712590f06245fbfba32d3be539812"  # version the code was checked against
!git clone -q https://github.com/1Tyson/Testing-And-Research-VJEPA.git -b $BRANCH /kaggle/working/repo
!git clone -q https://github.com/facebookresearch/vjepa2.git /kaggle/working/vjepa2 && cd /kaggle/working/vjepa2 && git checkout -q $VJEPA2_COMMIT
!pip install -q decord webdataset timm einops
import os
os.environ["VJEPA2_ROOT"] = "/kaggle/working/vjepa2"
R = "/kaggle/working/repo"
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# Official annotations (the full train csv is required: probe output classes are derived from it)
!git clone -q --depth 1 https://github.com/epic-kitchens/epic-kitchens-100-annotations.git /kaggle/working/ek100-ann
!ls /kaggle/input; find /kaggle/input -maxdepth 3 | head -30

In [ ]:
import glob, os
def find_video_root(root="/kaggle/input"):
    """Folder holding the EK100 videos: common parent of all .MP4 files among the attached inputs."""
    mp4 = glob.glob(os.path.join(root, "**", "*.MP4"), recursive=True)
    if not any(os.path.basename(p) == "P01_11.MP4" for p in mp4):
        raise RuntimeError("EK100 val videos not found: attach the video dataset via Add Input (e.g. download-ek100)")
    return os.path.commonpath(mp4)
VIDEO_ROOT = find_video_root()
print("VIDEO_ROOT =", VIDEO_ROOT)

!du -sh $VIDEO_ROOT
!find $VIDEO_ROOT -name "*.MP4" | wc -l

In [ ]:
# Resolution / bitrate of every val video, compared with EPIC_100_video_info.csv
!python $R/scripts/prepare_ek100.py --video_root $VIDEO_ROOT \
    --train_csv /kaggle/working/ek100-ann/EPIC_100_train.csv \
    --val_csv /kaggle/working/ek100-ann/EPIC_100_validation.csv \
    --out_dir /kaggle/working/work --probe_videos \
    --video_info /kaggle/working/ek100-ann/EPIC_100_video_info.csv > /kaggle/working/prepare.log
import json
rep = json.load(open("/kaggle/working/work/dataset_report.json"))
print("resolution:", rep.get("resolution_histogram"))
print("median bitrate (Mbit/s):", rep.get("median_bitrate_mbps"))
print("videos differing from official video info:", len(rep.get("mismatch_vs_official_video_info", {})))
print("warnings:", *rep["warnings"], sep="\n  ")

In [ ]:
# Our existing logits scored with the counting of the earlier lot-based notebook (reported 31.24)
prev = glob.glob("/kaggle/input/**/logits/official", recursive=True)
if prev:
    !mkdir -p /kaggle/working/logits && cp -r {prev[0]} /kaggle/working/logits/
    !python $R/scripts/diagnose_ek100.py --work_dir /kaggle/working/work --logits_dir /kaggle/working/logits/official \
        --bootstrap 20 --out /kaggle/working/diagnose.json | grep -E '"clean"|official_64gpu"|lots_of_15'
else:
    print("logits/official not attached: skip")